# 07 — Allocation: the closed loop (request → allocate → approve → verify)

The missing half of the thesis: *forecasts drive real capacity decisions*. Concept: a branch's
agent detects sustained pressure → raises a structured request → its **board (ICB)** ranks all
requests under a budget → funds what fits → next cycle shows whether it helped.

**Pre-registered rules (all stated, none tuned on outcomes):**
- **Forecasts**: the federated plane's own models (fed + fine-tuned, from notebook 04),
  rolled out 12 months recursively (predictions feed back as lags; backlog/throughput context
  frozen at last observed values — stated simplification).
- **Request rule (sustained pressure)**: unit RED in ≥ 2 of the last 3 observed months AND
  a positive 12-month capacity gap (forecast demand − recent throughput).
- **Board budget**: each ICB can fund extra capacity worth **2% of its trusts' current monthly
  throughput** (scarcity assumption — allocation is meaningless without it).
- **Priority**: gap × (1 + share of >52-week waiters) — worst long-waits first. Greedy fill;
  the last request may be partially funded.
- **Verification**: the whole workflow is backtested as-of **2025-03** and judged against the
  12 months of actuals we already hold.

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# operational series (same build as notebook 05)
df = pd.read_csv("../data/processed/rtt_long.csv")
net = json.load(open("../config/nhs_network.json"))
icb_of = {t["code"]: t["icb"] for r in net["regions"] for t in r["trusts"]}
key = ["provider_code", "specialty_code", "month"]
ser = ["provider_code", "specialty_code"]
cnt = df.pivot_table(index=key, columns="part", values="total_all", aggfunc="sum")
w = pd.DataFrame({"demand": cnt["Part_3"], "backlog": cnt["Part_2"],
                  "throughput": cnt["Part_1A"].fillna(0) + cnt["Part_1B"].fillna(0)}).reset_index()
bands = df[df["part"] == "Part_2"].groupby(key)[["weeks_0_18", "weeks_18_52", "weeks_52_plus"]].sum()
w = w.merge(bands.reset_index(), on=key, how="left").fillna(0)
banded = (w["weeks_0_18"] + w["weeks_18_52"] + w["weeks_52_plus"]).clip(lower=1)
w["share18"] = (w["weeks_18_52"] + w["weeks_52_plus"]) / banded
w["share52"] = w["weeks_52_plus"] / banded
months = sorted(w["month"].unique())
ok = w[w["demand"] > 0].groupby(ser)["month"].nunique()
w = w.set_index(ser).loc[ok[ok == len(months)].index].reset_index()
w = w.sort_values(ser + ["month"]).reset_index(drop=True)

# monthly flags (same constitution-based rule as notebook 05), computed for EVERY month
w["netflow3"] = w.groupby(ser, sort=False)["demand"].transform(lambda s: s.rolling(3).mean()) - \
                w.groupby(ser, sort=False)["throughput"].transform(lambda s: s.rolling(3).mean())
w["red"] = (w["share18"] > 0.08) & (w["netflow3"] > 0)

# federated models + per-series normalization stats (from notebooks 04 / 03)
feat = pd.read_csv("../data/processed/features.csv")
feat["spec"] = None
saved = torch.load("../data/processed/models_04.pt", weights_only=False)
specs, FEATS = saved["specs"], saved["feats"]
spec_idx = {s: i for i, s in enumerate(specs)}
device = "cuda" if torch.cuda.is_available() else "cpu"

class Net(nn.Module):
    def __init__(self, emb=4, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(len(specs), emb)
        self.mlp = nn.Sequential(nn.Linear(len(FEATS) + emb, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x, s):
        return x[:, 0] + self.mlp(torch.cat([x, self.emb(s)], dim=1)).squeeze(-1)

print(f"series: {w.groupby(ser).ngroups} | months {months[0]}..{months[-1]} | models: {len(saved['fed_ft'])} fine-tuned")

In [ ]:
# 12-month recursive rollout with the trust's fine-tuned federated model.
# Lags chain on predictions (standardized log space); lag12 comes from history;
# backlog/throughput context frozen at the last observed standardized values.
@torch.no_grad()
def forecast_12m(trust, as_of):
    sub = feat[(feat["provider_code"] == trust) & (feat["month"] <= as_of)]
    hist = sub.pivot(index="month", columns="specialty_code", values="y").sort_index()
    last = sub[sub["month"] == sub["month"].max()].set_index("specialty_code")
    cols = list(hist.columns)
    mu, sd = last["mu"].reindex(cols).values, last["sd"].reindex(cols).values
    ctx = last[["backlog", "throughput"]].reindex(cols).values  # frozen standardized context
    sidx = torch.tensor([spec_idx[c] for c in cols]).to(device)
    model = Net().to(device)
    model.load_state_dict(saved["fed_ft"][trust])
    model.eval()

    buf = hist.values  # (months, series) standardized log demand
    p = pd.Period(as_of, "M")
    preds = []
    for h in range(1, 13):
        tm = p + h
        lag12 = hist.loc[(tm - 12).strftime("%Y-%m")].values
        x = np.column_stack([buf[-1], buf[-2], buf[-3], lag12, ctx[:, 0], ctx[:, 1],
                             np.full(len(cols), np.sin(2 * np.pi * tm.month / 12)),
                             np.full(len(cols), np.cos(2 * np.pi * tm.month / 12))])
        y = model(torch.tensor(x, dtype=torch.float32).to(device), sidx).cpu().numpy()
        buf = np.vstack([buf, y])
        preds.append(np.expm1(y * sd + mu))  # back to real referral units
    return pd.DataFrame(np.array(preds), columns=cols)  # rows = months ahead 1..12

trusts = sorted(w["provider_code"].unique())
print("rollout ready;", len(trusts), "trusts")

In [ ]:
# branch agents raise requests: sustained RED (>=2 of last 3 months) AND positive forecast gap
def make_requests(as_of):
    upto = [m for m in months if m <= as_of]
    last3, last6 = upto[-3:], upto[-6:]
    red2 = w[w["month"].isin(last3)].groupby(ser)["red"].sum() >= 2
    thr6 = w[w["month"].isin(last6)].groupby(ser)["throughput"].mean()
    s52 = w[w["month"] == as_of].set_index(ser)["share52"]
    rows = []
    for t in trusts:
        fc = forecast_12m(t, as_of)
        for spec in fc.columns:
            if not red2.get((t, spec), False):
                continue
            gap = fc[spec].mean() - thr6[(t, spec)]
            if gap <= 0:
                continue
            rows.append({"provider_code": t, "specialty_code": spec, "icb": icb_of[t],
                         "gap": round(gap), "share52": s52[(t, spec)],
                         "priority": gap * (1 + s52[(t, spec)])})
    return pd.DataFrame(rows).sort_values("priority", ascending=False)

# boards allocate: budget = 2% of the ICB's current monthly throughput, greedy by priority
def allocate(requests, as_of, budget_pct=0.02):
    thr6 = w[w["month"].isin([m for m in months if m <= as_of][-6:])] \
        .groupby("provider_code")["throughput"].sum() / 6
    budgets = {icb: budget_pct * sum(thr6[t] for t in trusts if icb_of[t] == icb)
               for icb in set(icb_of.values())}
    out = []
    for _, r in requests.iterrows():
        grant = min(r["gap"], budgets[r["icb"]])
        budgets[r["icb"]] -= grant
        out.append({**r, "granted": round(grant),
                    "status": "FUNDED" if grant >= r["gap"] - 0.5 else ("PARTIAL" if grant > 0 else "DEFERRED")})
    return pd.DataFrame(out)

requests = make_requests(months[-1])
board = allocate(requests, months[-1])
print(f"as-of {months[-1]}: {len(board)} requests | "
      f"{(board['status'] == 'FUNDED').sum()} funded, {(board['status'] == 'PARTIAL').sum()} partial, "
      f"{(board['status'] == 'DEFERRED').sum()} deferred | "
      f"granted {board['granted'].sum():,} of requested {board['gap'].sum():,} slots/month")
print("\nboard memo — top 12 by priority:")
print(board.head(12)[["icb", "provider_code", "specialty_code", "gap", "granted", "status"]].to_string(index=False))

In [ ]:
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
})
top = board.head(15).iloc[::-1]
labels = top["provider_code"] + " · " + top["specialty_code"] + "  (" + top["icb"] + ")"
colors = top["status"].map({"FUNDED": "#2a78d6", "PARTIAL": "#eda100", "DEFERRED": "#898781"})
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(labels, top["gap"], color="#e1e0d9", height=0.65)          # requested
ax.barh(labels, top["granted"], color=colors, height=0.65)         # granted overlay
ax.set_title(f"Top capacity requests, {months[-1]} — granted (color) vs requested (gray)")
ax.set_xlabel("slots / month")
ax.grid(axis="y", visible=False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#2a78d6", "#eda100", "#898781"]],
          labels=["FUNDED", "PARTIAL", "DEFERRED"], frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# HQ strategic agent — the diagram's four decision types, each from a stated rule:
#   ALLOCATE = funded/partial this cycle (operational resource allocation)
#   ENHANCE  = chronic pressure (RED in >= 9 of last 12 months) -> invest in the service line
#   EXPAND   = trust whose DEFERRED gap exceeds 25% of its monthly throughput -> capacity expansion case
#   SEGREGATE = center-wise / region-wise views of where requested capacity concentrates
last12 = months[-12:]
red12 = w[w["month"].isin(last12)].groupby(ser)["red"].sum()
board["red12"] = [red12.get(i, 0) for i in zip(board["provider_code"], board["specialty_code"])]
board["decision"] = np.where(board["status"] != "DEFERRED", "ALLOCATE",
                    np.where(board["red12"] >= 9, "ENHANCE (invest in service line)", "monitor"))
print("decision mix:")
print(board["decision"].value_counts().to_string())

thr_trust = w[w["month"].isin(months[-6:])].groupby("provider_code")["throughput"].sum() / 6
deferred_gap = board[board["status"] == "DEFERRED"].groupby("provider_code")["gap"].sum()
expand = deferred_gap / thr_trust.reindex(deferred_gap.index) > 0.25
print("\nEXPANSION candidates (deferred gap > 25% of trust monthly throughput):")
print(deferred_gap[expand].astype(int).to_string() if expand.any() else "none this cycle")

reg_of = {t["code"]: r["region"] for r in net["regions"] for t in r["trusts"]}
board["region"] = board["provider_code"].map(reg_of)
print("\nregion-wise segregation of requested vs granted capacity (slots/month):")
print(board.groupby("region")[["gap", "granted"]].sum().astype(int).to_string())
print("\ncenter-wise (top 8 trusts by requested):")
print(board.groupby("provider_code")[["gap", "granted"]].sum().astype(int)
      .sort_values("gap", ascending=False).head(8).to_string())

## Verification — backtest the whole workflow as-of 2025-03

Run the identical pipeline a year ago, then judge it against what actually happened in
2025-04 → 2026-03:
- **request precision** — of the units the agents asked funding for, how many *actually* had
  demand exceed their capacity over the next 12 months?
- **forecast quality on requested units** — federated forecast vs the actual 12-month mean,
  compared against the naive (last observed month) forecast.

In [ ]:
AS_OF = "2025-03"
req_bt = make_requests(AS_OF)
board_bt = allocate(req_bt, AS_OF)

future = w[w["month"] > AS_OF]
actual_demand = future.groupby(ser)["demand"].mean()
thr6_bt = w[w["month"].isin([m for m in months if m <= AS_OF][-6:])].groupby(ser)["throughput"].mean()
last_obs = w[w["month"] == AS_OF].set_index(ser)["demand"]

idx = list(zip(board_bt["provider_code"], board_bt["specialty_code"]))
board_bt["actual_need"] = [actual_demand[i] - thr6_bt[i] for i in idx]
funded = board_bt[board_bt["status"] != "DEFERRED"]
precision = (funded["actual_need"] > 0).mean()
print(f"backtest as-of {AS_OF}: {len(board_bt)} requests, {len(funded)} funded/partial")
print(f"request precision: {precision:.0%} of funded units truly had demand > capacity over the next 12 months")

# forecast quality on requested units: federated rollout vs naive, against actual 12-month mean
fc_cache = {t: forecast_12m(t, AS_OF) for t in board_bt['provider_code'].unique()}
fed_fc = np.array([fc_cache[t][s].mean() for t, s in idx])
naive_fc = np.array([last_obs[i] for i in idx])
actual = np.array([actual_demand[i] for i in idx])
print(f"12-month-mean demand MAE on requested units: federated {np.abs(fed_fc - actual).mean():.0f} "
      f"vs naive {np.abs(naive_fc - actual).mean():.0f} referrals")

**The loop is closed.** Federated models (04) → forecasts → sustained-pressure requests →
board allocation under budget → human-readable memo → backtested verification. In the group
concept this is exactly: *ART @ Delhi sees rising demand → requests slots from HQ → the board
funds it if it ranks high enough under the budget → next year's data shows whether the call
was right.* Thresholds (8% standard, 2-of-3 sustained rule, 2% budget) are stated assumptions —
sensitivity analysis over them is A100-adjacent future work.